# 03 - Gold Dimensional Model

## Purpose

Build a business-ready Gold star schema for Retail Banking Customer Analytics.

## Concepts covered

- Dimensional modeling
- Surrogate keys and natural keys
- Fact and dimension table design
- Date dimension generation
- Fact-to-dimension joins
- Gold layer row count validation
- Sample analytical joins

## Prerequisites

02_silver_transformation.ipynb completed successfully and Silver Delta tables exist.

## Expected output

Dimensions: dim_customer, dim_account, dim_product, dim_branch, dim_date. Fact table: fact_transaction.

In [ ]:
from datetime import date, datetime, timezone
from typing import Iterable

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SILVER_TABLES = ["silver_customers", "silver_accounts", "silver_products", "silver_branches", "silver_transactions"]
GOLD_TABLES = ["dim_customer", "dim_account", "dim_product", "dim_branch", "dim_date", "fact_transaction"]

def log_info(message: str) -> None:
    print(f"[INFO] {datetime.now(timezone.utc).isoformat()} | {message}")

def require_table(table_name: str) -> DataFrame:
    if not spark.catalog.tableExists(table_name):
        raise RuntimeError(f"Required table is missing: {table_name}")
    return spark.table(table_name)

def add_surrogate_key(df: DataFrame, key_name: str, order_columns: Iterable[str]) -> DataFrame:
    window_spec = Window.orderBy(*[F.col(column_name).asc_nulls_last() for column_name in order_columns])
    return df.withColumn(key_name, F.row_number().over(window_spec).cast("long"))

def write_gold(df: DataFrame, table_name: str) -> int:
    row_count = df.count()
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)
    log_info(f"Wrote {table_name} with {row_count} rows")
    return row_count

for table_name in SILVER_TABLES:
    log_info(f"{table_name}: {require_table(table_name).count()} rows")

## Load Silver tables

In [ ]:
customers = require_table("silver_customers")
accounts = require_table("silver_accounts")
products = require_table("silver_products")
branches = require_table("silver_branches")
transactions = require_table("silver_transactions")

## Build dimensions

In [ ]:
dim_customer = (
    customers
    .select("customer_id", "first_name", "last_name", "email", "phone", "city", "state", "country", "customer_segment", "customer_status", "date_of_birth", "created_date", "is_active_customer")
    .withColumn("customer_full_name", F.concat_ws(" ", F.col("first_name"), F.col("last_name")))
    .withColumn("customer_age_years", F.floor(F.months_between(F.current_date(), F.col("date_of_birth")) / F.lit(12)))
    .withColumn("customer_tenure_years", F.round(F.months_between(F.current_date(), F.col("created_date")) / F.lit(12), 2))
)
dim_customer = add_surrogate_key(dim_customer, "customer_key", ["customer_id"])
write_gold(dim_customer, "dim_customer")

dim_product = products.select("product_id", "product_name", "product_category", "product_family", "fee_model", "is_active", "launch_date")
dim_product = add_surrogate_key(dim_product, "product_key", ["product_id"])
write_gold(dim_product, "dim_product")

dim_branch = branches.select("branch_id", "branch_name", "city", "state", "region", "opened_date", "is_active")
dim_branch = add_surrogate_key(dim_branch, "branch_key", ["branch_id"])
write_gold(dim_branch, "dim_branch")

dim_account = (
    accounts
    .select("account_id", "customer_id", "product_id", "branch_id", "account_type", "account_status", "opened_date", "closed_date", "current_balance", "is_open_account")
    .withColumn("account_age_years", F.round(F.months_between(F.current_date(), F.col("opened_date")) / F.lit(12), 2))
)
dim_account = add_surrogate_key(dim_account, "account_key", ["account_id"])
write_gold(dim_account, "dim_account")

## Build date dimension

In [ ]:
min_max_dates = transactions.agg(F.min("transaction_date").alias("min_date"), F.max("transaction_date").alias("max_date")).collect()[0]

if min_max_dates["min_date"] is None or min_max_dates["max_date"] is None:
    raise RuntimeError("Cannot build dim_date because silver_transactions has no transaction dates")

start_date = date(min_max_dates["min_date"].year, 1, 1)
end_date = date(min_max_dates["max_date"].year, 12, 31)

dim_date = (
    spark.sql(f"SELECT explode(sequence(to_date('{start_date}'), to_date('{end_date}'), interval 1 day)) AS date_value")
    .withColumn("date_key", F.date_format(F.col("date_value"), "yyyyMMdd").cast("int"))
    .withColumn("year", F.year(F.col("date_value")))
    .withColumn("quarter", F.quarter(F.col("date_value")))
    .withColumn("month", F.month(F.col("date_value")))
    .withColumn("month_name", F.date_format(F.col("date_value"), "MMMM"))
    .withColumn("year_month", F.date_format(F.col("date_value"), "yyyy-MM"))
    .withColumn("day_of_month", F.dayofmonth(F.col("date_value")))
    .withColumn("day_name", F.date_format(F.col("date_value"), "EEEE"))
    .withColumn("week_of_year", F.weekofyear(F.col("date_value")))
    .withColumn("is_weekend", F.dayofweek(F.col("date_value")).isin([1, 7]))
)

write_gold(dim_date, "dim_date")
display(dim_date.orderBy("date_key").limit(10))

## Build transaction fact table

In [ ]:
fact_transaction = (
    transactions.alias("t")
    .join(dim_account.alias("a"), F.col("t.account_id") == F.col("a.account_id"), "left")
    .join(dim_customer.alias("c"), F.col("a.customer_id") == F.col("c.customer_id"), "left")
    .join(dim_product.alias("p"), F.col("t.product_id") == F.col("p.product_id"), "left")
    .join(dim_branch.alias("b"), F.col("t.branch_id") == F.col("b.branch_id"), "left")
    .join(dim_date.alias("d"), F.col("t.transaction_date") == F.col("d.date_value"), "left")
    .select(
        F.col("t.transaction_id"), F.col("d.date_key"), F.col("c.customer_key"), F.col("a.account_key"), F.col("p.product_key"), F.col("b.branch_key"),
        F.col("t.account_id"), F.col("a.customer_id"), F.col("t.product_id"), F.col("t.branch_id"),
        F.col("t.transaction_timestamp"), F.col("t.transaction_date"), F.col("t.year_month"),
        F.col("t.transaction_type"), F.col("t.transaction_channel"), F.col("t.transaction_status"), F.col("t.currency"),
        F.col("t.transaction_amount"), F.col("t.absolute_transaction_amount"), F.col("t.debit_credit_indicator"), F.col("t.is_posted"),
        F.lit(1).alias("transaction_count"),
        (F.col("t.product_id") == F.col("a.product_id")).alias("product_matches_account"),
        (F.col("t.branch_id") == F.col("a.branch_id")).alias("branch_matches_account"),
        F.current_timestamp().alias("_gold_load_timestamp"),
    )
)

write_gold(fact_transaction, "fact_transaction")
display(fact_transaction.orderBy("transaction_timestamp"))

## Validate the Gold model

In [ ]:
validation_rows = [{"table_name": table_name, "row_count": spark.table(table_name).count()} for table_name in GOLD_TABLES]
row_count_df = spark.createDataFrame(validation_rows)
display(row_count_df.orderBy("table_name"))

silver_transaction_count = spark.table("silver_transactions").count()
gold_transaction_count = spark.table("fact_transaction").count()
log_info(f"Silver transactions: {silver_transaction_count}")
log_info(f"Gold fact transactions: {gold_transaction_count}")

if silver_transaction_count != gold_transaction_count:
    raise RuntimeError("Fact row count does not match silver_transactions row count")

null_key_summary = spark.sql("""
    SELECT
        SUM(CASE WHEN date_key IS NULL THEN 1 ELSE 0 END) AS missing_date_key,
        SUM(CASE WHEN customer_key IS NULL THEN 1 ELSE 0 END) AS missing_customer_key,
        SUM(CASE WHEN account_key IS NULL THEN 1 ELSE 0 END) AS missing_account_key,
        SUM(CASE WHEN product_key IS NULL THEN 1 ELSE 0 END) AS missing_product_key,
        SUM(CASE WHEN branch_key IS NULL THEN 1 ELSE 0 END) AS missing_branch_key
    FROM fact_transaction
""")
display(null_key_summary)

missing_key_total = sum(row_value for row_value in null_key_summary.collect()[0].asDict().values() if row_value is not None)
if missing_key_total > 0:
    raise RuntimeError(f"Gold fact has {missing_key_total} missing dimension key values")

## Sample business join

In [ ]:
branch_product_activity = spark.sql("""
    SELECT
        d.year_month,
        b.region,
        b.branch_name,
        p.product_category,
        COUNT(DISTINCT f.transaction_id) AS transaction_count,
        SUM(f.absolute_transaction_amount) AS total_transaction_amount
    FROM fact_transaction f
    JOIN dim_date d ON f.date_key = d.date_key
    JOIN dim_branch b ON f.branch_key = b.branch_key
    JOIN dim_product p ON f.product_key = p.product_key
    WHERE f.is_posted = true
    GROUP BY d.year_month, b.region, b.branch_name, p.product_category
    ORDER BY d.year_month, total_transaction_amount DESC
""")

display(branch_product_activity)

## Next steps

Run 04_data_quality_checks.ipynb to apply configured data quality rules before broad consumption.